## JAX: Accelerated Linear Algebra

JAX is a library developed by Google Research for high-performance numerical computing.

It provides a familiar NumPy-like API but adds powerful "transformations" like automatic differentiation (`grad`) and JIT compilation (`jit`).

While PyTorch is **Object-Oriented** (you call methods like `.backward()` on tensors), JAX is **Functional**. You treat your code as pure functions and use JAX to transform those functions into new ones (e.g., transforming a function into its derivative).

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

### Basic NumPy-like functionality

Most of the NumPy API is mirrored in `jax.numpy` (usually imported as `jnp`).

In [ ]:
print(jnp.linspace(-2, 2, 21))
print(jnp.ones((3, 3)))
print(jnp.zeros((2, 2)))

In [ ]:
x1 = jnp.eye(3)
x2 = jnp.reshape(jnp.arange(1, 10), (3, 3))

# Familiar math operations
result = jnp.sin(x1) + jnp.exp(x2) * (x1 + 1) / (x2**2)
result

### Mixing JAX and NumPy

JAX arrays (Tensors) and NumPy arrays are highly compatible. You can pass JAX arrays to many NumPy functions, and JAX functions will automatically convert NumPy inputs into JAX arrays.

In [ ]:
x_np = np.ones(3)
x_jnp = jnp.arange(3.0)

# JAX handles numpy arrays seamlessly
print(f"Addition: {x_jnp + x_np}")

# To explicitly get a numpy array back, use np.array()
back_to_np = np.array(x_jnp)
print(f"Type: {type(back_to_np)}")

### Automatic Differentiation: The `grad` Transformation

In JAX, you don't call a method on a variable to find its gradient. Instead, you use `jax.grad()` to create a **gradient function**.

In [ ]:
def f_jax(x):
    return -jnp.exp(-(x[0]**2 + x[1]**2))

# Transform f into its derivative df
df_jax = jax.grad(f_jax)

test_point = jnp.array([0.2, -0.3])
print(f"Function value: {f_jax(test_point)}")
print(f"Gradient value: {df_jax(test_point)}")

### Building Gradient Descent with JAX

Because JAX arrays are **immutable** (meaning you cannot change them in place like `x += 1`), the update step looks slightly different from PyTorch.

In [ ]:
def grad_desc_jax(f, x0, epsilon=1e-3, T=200, alpha=0.1):
    df = jax.grad(f)
    x = jnp.array(x0)
    trace = []
    
    for i in range(T):
        dfdx = df(x)
        err = jnp.max(jnp.abs(dfdx))
        
        status = {
            "i": i,
            "x": np.array(x),
            "err": float(err)
        }
        trace.append(status)
        
        if err < epsilon:
            return trace
        
        # Update: We create a new array for x each step
        x = x - alpha * dfdx
        
    return trace

trace = grad_desc_jax(f_jax, [2.0, -0.3])
print(f"Final result after {len(trace)} iterations: {trace[-1]['x']}")

### Optimization with JIT

One of JAX's most powerful features is **JIT (Just-In-Time) compilation**. This takes your Python/JAX code and compiles it into optimized machine code (XLA) that runs significantly faster.

In [ ]:
# Compiling the gradient function
df_fast = jax.jit(jax.grad(f_jax))

# The first call is slow (compiling), subsequent calls are very fast
print(df_fast(test_point))